In [6]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score
from wittgenstein import RIPPER
from imodels import RuleFitClassifier


In [ ]:
# --- Setup ---
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

n_splits = 20
splitter = StratifiedShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=42)

ripper_rules_all = []
ripper_accuracies = []
rulefit_rules_all = []
rulefit_accuracies = []

# --- Run both methods across 20 splits ---
for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    # RIPPER
    rip = RIPPER(k=2)
    rip.fit(X_tr, y_tr, class_feat='target')
    ripper_preds = rip.predict(X_te)
    ripper_accuracies.append(accuracy_score(y_te, ripper_preds))
    for rule in rip.ruleset_:
        ripper_rules_all.append(str(rule))

    # RuleFit
    rf = RuleFitClassifier(max_rules=15, random_state=fold)
    rf.fit(X_tr.values, y_tr, feature_names=list(data.feature_names))
    rulefit_preds = rf.predict(X_te.values)
    rulefit_accuracies.append(accuracy_score(y_te, rulefit_preds))

    rules_df = rf._get_rules()
    active = rules_df[(rules_df['type'] == 'rule') & (rules_df['coef'].abs() > 0.01)]
    rulefit_rules_all.extend(active['rule'].tolist())

print(f"RIPPER accuracy:  {np.mean(ripper_accuracies):.3f} +/- {np.std(ripper_accuracies):.3f}")
print(f"RuleFit accuracy: {np.mean(rulefit_accuracies):.3f} +/- {np.std(rulefit_accuracies):.3f}")
print()

# --- RIPPER feature frequency ---
nospace_to_real = {feat.replace(' ', ''): feat for feat in data.feature_names}

ripper_feature_counts = Counter()
for rule_str in ripper_rules_all:
    tokens = re.findall(r'([a-z]+)(?==)', rule_str)
    for tok in tokens:
        if tok in nospace_to_real:
            ripper_feature_counts[nospace_to_real[tok]] += 1

print("=== RIPPER Feature Frequency Across 20 Splits ===")
print(f"Total RIPPER rules collected: {len(ripper_rules_all)}")
total_uses_rip = sum(ripper_feature_counts.values())
print(f"Total feature uses (a rule may reference the same feature in multiple conditions): {total_uses_rip}\n")
for feat, count in ripper_feature_counts.most_common(10):
    pct = count / total_uses_rip * 100 if total_uses_rip else 0
    print(f"  {feat}: {count} uses ({pct:.0f}% of all uses)")
print()

# --- RuleFit feature frequency ---
rulefit_feature_counts = Counter()
for rule_str in rulefit_rules_all:
    feats = re.findall(r'([a-z][a-z ]*?)\s*(?:<=|>)\s*[\-\d.]', rule_str)
    for f in feats:
        f = f.strip()
        if f in data.feature_names:
            rulefit_feature_counts[f] += 1

print("=== RuleFit Feature Frequency Across 20 Splits ===")
print(f"Total active RuleFit rules collected: {len(rulefit_rules_all)}")
total_uses_rf = sum(rulefit_feature_counts.values())
print(f"Total feature uses: {total_uses_rf}\n")
for feat, count in rulefit_feature_counts.most_common(10):
    pct = count / total_uses_rf * 100 if total_uses_rf else 0
    print(f"  {feat}: {count} uses ({pct:.0f}% of all uses)")

RIPPER accuracy:  0.898 +/- 0.023
RuleFit accuracy: 0.943 +/- 0.020

=== RIPPER Feature Frequency Across 20 Splits ===
Total RIPPER rules collected: 287
Total feature uses (a rule may reference the same feature in multiple conditions): 386

  mean concave points: 98 uses (25% of all uses)
  worst concave points: 28 uses (7% of all uses)
  mean radius: 28 uses (7% of all uses)
  worst concavity: 27 uses (7% of all uses)
  mean concavity: 25 uses (6% of all uses)
  worst perimeter: 24 uses (6% of all uses)
  worst radius: 23 uses (6% of all uses)
  worst area: 17 uses (4% of all uses)
  worst texture: 16 uses (4% of all uses)
  mean texture: 14 uses (4% of all uses)

=== RuleFit Feature Frequency Across 20 Splits ===
Total active RuleFit rules collected: 268
Total feature uses: 268

  worst area: 58 uses (22% of all uses)
  area error: 40 uses (15% of all uses)
  worst texture: 35 uses (13% of all uses)
  mean texture: 26 uses (10% of all uses)
  worst radius: 25 uses (9% of all uses)
  

In [10]:
print(X[['mean concave points', 'worst concave points']].corr())
print(X[['worst area', 'area error', 'worst radius', 'worst perimeter']].corr())

                      mean concave points  worst concave points
mean concave points              1.000000              0.910155
worst concave points             0.910155              1.000000
                 worst area  area error  worst radius  worst perimeter
worst area         1.000000    0.811408      0.984015         0.977578
area error         0.811408    1.000000      0.757373         0.761213
worst radius       0.984015    0.757373      1.000000         0.993708
worst perimeter    0.977578    0.761213      0.993708         1.000000
